In [ ]:
import tensorflow as tf
print("GPU 列表:", tf.config.list_physical_devices('GPU'))

In [ ]:
import os

In [ ]:
import tensorflow as tf
from tensorflow import keras

cifar10 = keras.datasets.cifar10
(X_train_full,y_train_full),(X_test,y_test) = cifar10.load_data()

X_train_full,X_test = X_train_full/255.0 ,X_test/255.0

In [ ]:
X_train,X_valid = X_train_full[5000:],X_train_full[:5000]
y_train,y_valid = y_train_full[5000:],y_train_full[:5000]

In [ ]:
X_train.shape,X_test.shape,y_train.shape,y_test.shape,X_valid.shape

In [ ]:
import tensorflow as tf
from tensorflow import keras

# 建造顺序模型大楼
model = keras.models.Sequential([
    keras.Input(shape=(32, 32, 3)),
    # 第 1 阶段：初级侦探部门
    keras.layers.Conv2D(filters=32, kernel_size=(3, 3), strides=1,activation='relu'),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),

    # 第 2 阶段：中级侦探部门
    keras.layers.Conv2D(filters=64, kernel_size=(3, 3), activation='relu',padding = "same"),
    keras.layers.Conv2D(filters=64, kernel_size=(3, 3), activation='relu',padding = "same"),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),

    # 第 3 阶段：高级侦探部门
    keras.layers.Conv2D(filters=128, kernel_size=(3, 3), activation='relu',padding = "same"),
    keras.layers.Conv2D(filters=128, kernel_size=(3, 3), activation='relu',padding = "same"),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),

    # 跨部门交接：拍扁成一维数据
    keras.layers.Flatten(),

    # 法官审判庭：全连接层
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(10, activation='softmax')
])

# 打印大楼的透视图
model.summary()

In [ ]:
from tensorflow.keras import optimizers
optimizer = optimizers.Adam(learning_rate = 0.0001)
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"]
)
#加入可视化

log_dir = os.path.join("logs", "fit")  
tensorboard_cb = keras.callbacks.TensorBoard(log_dir=log_dir)


callbacks=[
    keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("mnist_best.keras", save_best_only=True),
    tensorboard_cb
]

In [ ]:
model.fit(X_train,y_train,epochs = 100,
          validation_data=[X_valid,y_valid],
          callbacks=callbacks
         )

In [ ]:
#可视化结果
%load_ext tensorboard
%tensorboard --logdir logs/fit

In [ ]:
model.evaluate(X_test, y_test)

用fashion mnist的数据试一下这个model


In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

fashion_mnist = keras.datasets.fashion_mnist
(X_train_full_F, y_train_full_F), (X_test_F, y_test_F) = fashion_mnist.load_data()

X_valid_F = (X_train_full_F[:5000] / 255.0).astype(np.float32)
X_train_F = (X_train_full_F[5000:] / 255.0).astype(np.float32)
y_valid_F = y_train_full_F[:5000]
y_train_F = y_train_full_F[5000:]

X_train_F = np.expand_dims(X_train_F, axis=-1)
X_valid_F = np.expand_dims(X_valid_F, axis=-1)
#下边的和上边的一样的意思
#X_train = X_train[..., np.newaxis]
#X_valid = X_valid[..., np.newaxis]
#X_test = X_test[..., np.newaxis]

# 2. 建造大楼！有BN层
model = keras.models.Sequential([
    keras.layers.Conv2D(64, 7, activation="relu", padding="same", input_shape=[28, 28, 1]),
    keras.layers.BatchNormalization(), # 🌟 超级减震器！强行稳定数值
    keras.layers.MaxPooling2D(2),
    
    keras.layers.Conv2D(128, 3, activation="relu", padding="same"),
    keras.layers.BatchNormalization(), # 🌟
    keras.layers.Conv2D(128, 3, activation="relu", padding="same"),
    keras.layers.BatchNormalization(), # 🌟
    keras.layers.MaxPooling2D(2),
    
    keras.layers.Conv2D(256, 3, activation="relu", padding="same"),
    keras.layers.BatchNormalization(), # 🌟
    keras.layers.Conv2D(256, 3, activation="relu", padding="same"),
    keras.layers.BatchNormalization(), # 🌟
    keras.layers.MaxPooling2D(2),
    
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.BatchNormalization(), # 🌟
    keras.layers.Dropout(0.5),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.BatchNormalization(), # 🌟
    keras.layers.Dropout(0.5),
    keras.layers.Dense(10, activation="softmax") 
])

# 3. 编译 (有了减震器，直接用 Adam！)
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="adam", 
              metrics=["accuracy"])

early_stopping_cb = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

history = model.fit(X_train_F, y_train_F, epochs=30,
                    validation_data=(X_valid_F, y_valid_F),
                    callbacks=[early_stopping_cb])

In [ ]:
X_test_F_clean = (X_test_F / 255.0).astype(np.float32)
X_test_F_clean = np.expand_dims(X_test_F_clean, axis=-1)

In [ ]:
print("成绩揭晓：")
model.evaluate(X_test_F_clean, y_test_F)

In [ ]:
import tensorflow as tf
from tensorflow import keras

input_layer = keras.layers.Input(shape=(28, 28, 1), name="Input_Door")

# 为了演示，我们先加一层普通的卷积作为前台大厅
x = keras.layers.Conv2D(64, 7, activation="relu", padding="same")(input_layer)
x = keras.layers.MaxPooling2D(2)(x)

# inception模块
branch_1 = keras.layers.Conv2D(64, 1, activation="relu", padding="same")(x)

branch_2 = keras.layers.Conv2D(96, 1, activation="relu", padding="same")(x)
branch_2 = keras.layers.Conv2D(128, 3, activation="relu", padding="same")(branch_2)

branch_3 = keras.layers.Conv2D(16, 1, activation="relu", padding="same")(x)
branch_3 = keras.layers.Conv2D(32, 5, activation="relu", padding="same")(branch_3)

branch_4 = keras.layers.MaxPooling2D(3, strides=1, padding="same")(x)
branch_4 = keras.layers.Conv2D(32, 1, activation="relu", padding="same")(branch_4)

inception_output = keras.layers.concatenate([branch_1, branch_2, branch_3, branch_4], axis=-1)
#inception模块结束

x = keras.layers.GlobalAveragePooling2D()(inception_output)
x = keras.layers.Dropout(0.4)(x)
output_layer = keras.layers.Dense(10, activation="softmax")(x)


mini_googlenet = keras.Model(inputs=input_layer, outputs=output_layer)

# mini_googlenet.summary()

<span style="color:red">一个inception的结构</span>
```python

import tensorflow as tf
from tensorflow import keras

def inception_module(x, filters_1x1, filters_3x3_reduce, filters_3x3, filters_5x5_reduce, filters_5x5, filters_pool_proj):
    #分之1，卷积层
    branch_1 = keras.layers.Conv2D(filters_1x1, 1, activation="relu", padding="same")(x)
    #分之2，
    branch_2 = keras.layers.Conv2D(filters_3x3_reduce, 1, activation="relu", padding="same")(x)
    branch_2 = keras.layers.Conv2D(filters_3x3, 3, activation="relu", padding="same")(branch_2)
    
    branch_3 = keras.layers.Conv2D(filters_5x5_reduce, 1, activation="relu", padding="same")(x)
    branch_3 = keras.layers.Conv2D(filters_5x5, 5, activation="relu", padding="same")(branch_3)
    
    branch_4 = keras.layers.MaxPooling2D(3, strides=1, padding="same")(x)
    branch_4 = keras.layers.Conv2D(filters_pool_proj, 1, activation="relu", padding="same")(branch_4)
    #这个concatenate是将几个并行的通路，厚度叠加，其长宽是相同的。（也必须相同）
    output = keras.layers.concatenate([branch_1, branch_2, branch_3, branch_4], axis=-1)
    
    return output

```

<span style="color:red">残差块的定义函数</span>

```python 
import tensorflow as tf
from tensorflow import keras

def residual_block(x, filters, strides=1):
    #x: 上一层传进来的数据 (原稿)
    #filters: 这一层输出的厚度
    #strides: 步幅 (如果是2，图片长宽会被缩小一半)
    shortcut = x

    # 🌟如果主干道缩小了图片，或者改变了厚度，
    # 原稿也必须用 1x1 的filters做一次“同步变形”！否则俩积木加不到一起！
    if strides != 1 or x.shape[-1] != filters:
        shortcut = keras.layers.Conv2D(filters, kernel_size=1, strides=strides, padding="same")(shortcut)
        shortcut = keras.layers.BatchNormalization()(shortcut)

    # 第一步：卷积 -> BN -> ReLU(activation)
    main_path = keras.layers.Conv2D(filters, kernel_size=3, strides=strides, padding="same")(x)
    main_path = keras.layers.BatchNormalization()(main_path)
    main_path = keras.layers.Activation("relu")(main_path)

    # 第二步：卷积 -> BN
    main_path = keras.layers.Conv2D(filters, kernel_size=3, strides=1, padding="same")(main_path)
    main_path = keras.layers.BatchNormalization()(main_path)
    # 🌟这里千万不能接 ReLU！ 应该先合并，在接！

    # 🌟 核心区别：Inception 是“拼接 (Concatenate)”，厚度相加；
    # 残差块是直接“数学相加 (Add)”，比如 原像素 100 + 残差修改量 -5 = 95！
    #也就是说，需要shortcut 和 main_path的shape完全相同。
    output = keras.layers.add([shortcut, main_path])

    # 最后，ReLU,输出残差块
    output = keras.layers.Activation("relu")(output)

    return output
```

<span style="color:red">定义一个函数，将inception和残差连接在一起</span>

```python 
import tensorflow as tf
from tensorflow import keras

def inception_res(x,filters_1x1, filters_3x3_reduce, filters_3x3, filters_5x5_reduce, filters_5x5, filters_pool_proj):
    shortcut = x

    branch_1 = keras.layers.Conv2D(filters_1x1, 1, activation="relu", padding="same")(x)
    #分之2，
    branch_2 = keras.layers.Conv2D(filters_3x3_reduce, 1, activation="relu", padding="same")(x)
    branch_2 = keras.layers.Conv2D(filters_3x3, 3, activation="relu", padding="same")(branch_2)
    
    branch_3 = keras.layers.Conv2D(filters_5x5_reduce, 1, activation="relu", padding="same")(x)
    branch_3 = keras.layers.Conv2D(filters_5x5, 5, activation="relu", padding="same")(branch_3)
    
    branch_4 = keras.layers.MaxPooling2D(3, strides=1, padding="same")(x)
    branch_4 = keras.layers.Conv2D(filters_pool_proj, 1, activation="relu", padding="same")(branch_4)
    #这个concatenate是将几个并行的通路，厚度叠加，其长宽是相同的。
    output = keras.layers.concatenate([branch_1, branch_2, branch_3, branch_4], axis=-1)

    y = x.shape[-1]
    output = keras.layers.Conv2D(y,1,padding="same")(output)
    
    output = keras.layers.BatchNormalization()(output)  #BN层
    
    output = keras.layers.add([shortcut,output])     #想加
    output = keras.layers.Activation("relu")(output)

    return output

 <span style="color:red">预训练的model，用别人的model来训练自己的数据</span>
```python
tf.random.set_seed(42)  # extra code – ensures reproducibility
base_model = tf.keras.applications.xception.Xception(weights="imagenet",
                                                     include_top=False)
                                                    #input_shape = (256,256,3),这个3 必须和你用的这个model的原始输入的一样。
#可以不要顶部（输出层），也可以不要底层（input）。只要加入input_shape = (),就可以了，改层自己的形状。
                                        #但是输入的通道数不能改，就是图片的大小能改，通道数不能改，

avg = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
#平均池化层
output = tf.keras.layers.Dense(n_classes, activation="softmax")(avg)
#output层
model = tf.keras.Model(inputs=base_model.input, outputs=output)
#model最终合体！


```
<span style="color:red">切记,先冻结这个model，训练加入的output层，先跑几轮，在去解冻一些层数，去完整跑一下</span>

关于输出带框的，就是确定物体位置的，就只需要，加上
```python
loc_output = tf.keras.layers.Dense(4)(avg)
```
#将位置输出的结果加入进去就好。<span style="color:red">但也别忘记了，修改compile</span>

```python
model = tf.keras.Model(inputs=base_model.input, 
                       outputs=[output,loc_output])
```